# 04. AI가 내 연주를 실시간 추적하다 — **협업**

## 이 노트북의 위치

```
기반     : NB01 (듣다)
확장     : NB02 (시각화)
협업 ✦   : NB04 (실시간 추적) ← 여기
통합     : NB05 (무대)
```

**확장**은 연주자가 만든 데이터를 새 차원(영상)으로 바꾸는 **일방향** 작업이었습니다.  
**협업**은 다릅니다 — AI가 내 연주를 **실시간으로 따라가고**, 연주자는 "잘 따라오는가?"를 판단합니다.

### matchmaker

**[matchmaker](https://github.com/pymatchmaker/matchmaker)**는 ISMIR 2025에서 발표된 오픈소스 악보 추적(score following) 라이브러리입니다.

- 악보(MIDI)와 연주(오디오)를 받으면, "지금 악보의 몇 번째 박자를 치고 있는가"를 실시간으로 알려줍니다.
- 루바토가 있어도, 빨라져도, 느려져도 따라갑니다.
- 이 위치 정보를 NB02의 시각화와 연결하면: **연주에 맞춰 영상이 동기화됩니다.**

**입력**: `assets/scores/*.mid` (원본 악보), `assets/*.wav` (연주 녹음)  
**출력**: `artifacts/alignment/*.json` (정렬 결과 → NB05 /stage에서 사용)

In [ ]:
# Colab용 설치 (서버 시연 환경에서는 건너뛰어도 됩니다)
!apt-get -qq install -y fluidsynth portaudio19-dev > /dev/null 2>&1
!pip install -q matchmaker pretty_midi numpy matplotlib

import warnings
warnings.filterwarnings('ignore')
print('✅ 설치 완료!')

---
## 1. 악보 추적이란?

피아니스트가 악보를 보며 연주합니다.  
옆에서 듣는 사람(또는 AI)이 **"지금 악보의 어디를 치고 있는지"** 실시간으로 파악하는 것 — 이것이 **악보 추적(Score Following)**입니다.

```
연주 오디오  ──▶  matchmaker  ──▶  "지금 악보의 17.3번째 박자"
     +                              ↓
악보 MIDI   ──▶                 시각화/페이지 넘기기/자동 반주
```

### 왜 어려운가?

- 메트로놈에 맞춰 치면 쉽겠지만, 실제 연주에는 **루바토**, **급가속**, **일시 정지**가 있습니다.
- Satie의 느린 루바토와 Prokofiev의 빠른 토카타 — AI가 둘 다 따라갈 수 있을까?
- 이것을 직접 확인하는 것이 이 노트북의 핵심입니다.

---
## 2. 오프라인 정렬 — 녹음으로 테스트

라이브 마이크 없이, **녹음 파일**을 악보와 정렬합니다.  
matchmaker에 `performance_file`을 넘기면 녹음을 실시간처럼 재생하며 정렬합니다.

In [ ]:
from pathlib import Path
import json
import numpy as np

ARTIFACTS = Path('artifacts')
ALIGNMENT_DIR = ARTIFACTS / 'alignment'
ALIGNMENT_DIR.mkdir(parents=True, exist_ok=True)

pieces = {
    'satie': {
        'title': 'Satie — Gymnopédie No.1',
        'score': 'assets/scores/satie_gymnopedie.mid',
        'audio': 'assets/satie_gymnopedie_no1.wav',
    },
    'prokofiev': {
        'title': 'Prokofiev — Toccata Op.11',
        'score': 'assets/scores/prokofiev_toccata.mid',
        'audio': 'assets/prokofiev_toccata_op11.wav',
    },
}

for key, p in pieces.items():
    for f in ['score', 'audio']:
        if not Path(p[f]).exists():
            raise FileNotFoundError(f"{p[f]} 없음")
    print(f"✅ {p['title']}")

print(f"\n정렬 결과 저장 위치: {ALIGNMENT_DIR}")

In [ ]:
from matchmaker import Matchmaker
import time

print("🎹 Satie — Gymnopédie No.1 정렬 시작...")
t0 = time.time()

mm_satie = Matchmaker(
    score_file=pieces['satie']['score'],
    performance_file=pieces['satie']['audio'],
    input_type="audio",
)

satie_positions = []
for pos in mm_satie.run():
    satie_positions.append(float(pos))

elapsed = time.time() - t0
print(f"✅ 완료! {len(satie_positions)}개 프레임, {elapsed:.1f}초 소요")
print(f"   악보 위치 범위: {min(satie_positions):.1f} ~ {max(satie_positions):.1f} beats")

In [ ]:
print("🎹 Prokofiev — Toccata Op.11 정렬 시작...")
t0 = time.time()

mm_prok = Matchmaker(
    score_file=pieces['prokofiev']['score'],
    performance_file=pieces['prokofiev']['audio'],
    input_type="audio",
)

prok_positions = []
for pos in mm_prok.run():
    prok_positions.append(float(pos))

elapsed = time.time() - t0
print(f"✅ 완료! {len(prok_positions)}개 프레임, {elapsed:.1f}초 소요")
print(f"   악보 위치 범위: {min(prok_positions):.1f} ~ {max(prok_positions):.1f} beats")

---
## 3. 정렬 결과 시각화

X축은 **연주 시간**(초), Y축은 **악보 위치**(beat).  
완벽한 정렬은 매끄러운 상승 곡선입니다. 루바토가 있으면 곡선이 늘어지고, 빠른 구간에서는 급경사가 됩니다.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (key, positions) in zip(axes, [('satie', satie_positions), ('prokofiev', prok_positions)]):
    n = len(positions)
    frame_rate = mm_satie.config.frame_rate if key == 'satie' else mm_prok.config.frame_rate
    times = np.arange(n) / frame_rate
    
    ax.plot(times, positions, linewidth=0.8, color='#4F46E5' if key == 'satie' else '#DC2626')
    ax.set_xlabel('연주 시간 (초)', fontsize=11)
    ax.set_ylabel('악보 위치 (beat)', fontsize=11)
    ax.set_title(pieces[key]['title'], fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(ALIGNMENT_DIR / 'alignment_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print("💾 alignment_curves.png 저장됨")

---
## 4. 두 곡 비교 — 정렬 난이도

같은 알고리즘이 두 곡을 어떻게 다르게 따라가는지 분석합니다.

In [ ]:
def analyze_alignment(positions, frame_rate):
    """정렬 곡선의 특성을 계산합니다."""
    pos = np.array(positions)
    velocity = np.diff(pos) * frame_rate  # beats/sec
    return {
        'total_frames': len(pos),
        'duration_sec': len(pos) / frame_rate,
        'beat_range': (float(pos.min()), float(pos.max())),
        'avg_tempo_bps': float(np.mean(velocity)),
        'tempo_std_bps': float(np.std(velocity)),
        'tempo_cv': float(np.std(velocity) / max(np.mean(velocity), 1e-6)),
        'max_speed_bps': float(np.max(np.abs(velocity))),
    }

fr_s = mm_satie.config.frame_rate
fr_p = mm_prok.config.frame_rate

stats_s = analyze_alignment(satie_positions, fr_s)
stats_p = analyze_alignment(prok_positions, fr_p)

print(f"{'':20s} {'Satie':>12s} {'Prokofiev':>12s}")
print(f"{'─'*46}")
print(f"{'연주 길이 (초)':20s} {stats_s['duration_sec']:12.1f} {stats_p['duration_sec']:12.1f}")
print(f"{'평균 속도 (beats/s)':20s} {stats_s['avg_tempo_bps']:12.2f} {stats_p['avg_tempo_bps']:12.2f}")
print(f"{'속도 편차 (σ)':20s} {stats_s['tempo_std_bps']:12.2f} {stats_p['tempo_std_bps']:12.2f}")
print(f"{'속도 변동계수 (CV)':20s} {stats_s['tempo_cv']:12.2f} {stats_p['tempo_cv']:12.2f}")
print(f"{'최대 순간 속도':20s} {stats_s['max_speed_bps']:12.2f} {stats_p['max_speed_bps']:12.2f}")
print()
if stats_s['tempo_cv'] > stats_p['tempo_cv']:
    print("→ Satie의 속도 변동이 더 큽니다 — 루바토가 많다는 뜻입니다.")
else:
    print("→ Prokofiev의 속도 변동이 더 큽니다 — 빠르고 불규칙한 구간이 있다는 뜻입니다.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (key, positions, fr) in zip(axes, [
    ('satie', satie_positions, fr_s),
    ('prokofiev', prok_positions, fr_p),
]):
    pos = np.array(positions)
    velocity = np.diff(pos) * fr
    window = min(50, len(velocity) // 4)
    if window > 0:
        smooth = np.convolve(velocity, np.ones(window)/window, mode='valid')
        times = np.arange(len(smooth)) / fr
    else:
        smooth = velocity
        times = np.arange(len(velocity)) / fr
    
    color = '#4F46E5' if key == 'satie' else '#DC2626'
    ax.plot(times, smooth, linewidth=0.8, color=color)
    ax.axhline(y=np.mean(velocity), color=color, linestyle='--', alpha=0.5, label=f'평균 {np.mean(velocity):.2f} bps')
    ax.set_xlabel('연주 시간 (초)', fontsize=11)
    ax.set_ylabel('추적 속도 (beats/sec)', fontsize=11)
    ax.set_title(f'{pieces[key]["title"]} — 속도 변화', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(ALIGNMENT_DIR / 'tempo_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print("💾 tempo_curves.png 저장됨")

### 핵심 질문

- **Satie**: 루바토 구간에서 AI가 잘 따라갔나요? 곡선이 부드럽게 늘어지면 성공입니다.
- **Prokofiev**: 빠른 패시지에서 AI가 놓치지 않았나요? 급경사 구간에서 끊김이 없으면 성공입니다.
- 어느 곡이 AI에게 더 어려웠을까요? 속도 변동계수(CV)가 힌트입니다.

---
## 5. /stage 연동 준비

정렬 결과를 JSON으로 저장합니다.  
NB05의 `/stage` 페이지는 이 데이터를 사용해 **악보 위치 → cue 프레임** 변환을 수행하고,  
matchmaker 서버에서 실시간으로 받은 beat position을 시각화와 동기화합니다.

In [ ]:
import pretty_midi

def build_beat_to_time_map(score_midi_path):
    """악보 MIDI에서 beat -> time(초) 변환 테이블 생성."""
    midi = pretty_midi.PrettyMIDI(score_midi_path)
    times = midi.get_beats()
    return [{'beat': float(i), 'time': float(t)} for i, t in enumerate(times)]

for key, p in pieces.items():
    positions = satie_positions if key == 'satie' else prok_positions
    fr = fr_s if key == 'satie' else fr_p
    
    beat_time_map = build_beat_to_time_map(p['score'])
    
    alignment_data = {
        'piece': key,
        'title': p['title'],
        'score_file': p['score'],
        'audio_file': p['audio'],
        'frame_rate': fr,
        'total_frames': len(positions),
        'positions': positions,
        'beat_to_time': beat_time_map,
    }
    
    out_path = ALIGNMENT_DIR / f'{key}_alignment.json'
    out_path.write_text(json.dumps(alignment_data, ensure_ascii=False))
    print(f"💾 {out_path} ({len(positions)} frames, {len(beat_time_map)} beats)")

print("\n✅ 정렬 결과가 저장되었습니다.")
print("→ NB05에서 /stage 페이지의 라이브 추적 모드와 연결합니다.")

---
## 6. 라이브 추적 서버 (시연용)

수업 시연에서는 서버에서 matchmaker를 실행하고, `/stage` 페이지가 WebSocket으로 실시간 위치를 받습니다.

```
마이크/오디오 → matchmaker 서버 → WebSocket → /stage 시각화
```

서버 실행 방법:
```bash
cd server
uvicorn main:app --host 0.0.0.0 --port 8765
```

서버 코드는 `server/main.py`에 있습니다.

In [ ]:
print("서버 연동 테스트 (선택 사항)")
print("="*40)
print()
print("1. 터미널에서 서버 실행:")
print("   cd server && uvicorn main:app --host 0.0.0.0 --port 8765")
print()
print("2. 아래 셀을 실행하면 서버에 테스트 요청을 보냅니다.")
print("   (서버가 실행 중이 아니면 건너뛰세요)")

In [ ]:
import asyncio

async def test_server():
    try:
        import websockets
    except ImportError:
        print("websockets 패키지가 없습니다. pip install websockets")
        return
    
    uri = "ws://localhost:8765/ws/follow"
    try:
        async with websockets.connect(uri) as ws:
            await ws.send(json.dumps({
                "score_file": pieces['satie']['score'],
                "performance_file": pieces['satie']['audio'],
            }))
            count = 0
            async for msg in ws:
                data = json.loads(msg)
                count += 1
                if count <= 5:
                    print(f"  프레임 {count}: beat = {data['beat']:.2f}")
                elif count == 6:
                    print(f"  ... (나머지 프레임 수신 중)")
            print(f"\n✅ 총 {count}개 프레임 수신 완료")
    except ConnectionRefusedError:
        print("❌ 서버에 연결할 수 없습니다. 서버가 실행 중인지 확인하세요.")
    except Exception as e:
        print(f"❌ 오류: {e}")

await test_server()

---
## 협업의 핵심

matchmaker는 **"지금 여기를 치고 있다"**고 응답합니다.  
연주자는 **"맞게 따라오는가? 내 루바토를 반영하는가?"**를 판단합니다.

이것이 AI와 연주자의 **진짜 협업**입니다:
- AI가 음을 만들어내는 게 아니라
- AI가 내 연주를 **이해하고 반응**하고
- 연주자가 그 반응을 **평가하고 조정**합니다.

---

**→ NB05 (`05_stage.ipynb`)**: 모든 산출물을 하나의 무대로 엮습니다. `/stage` 페이지에서 라이브 추적 모드를 시연합니다.